In [109]:
#to prevent colab to automatically disconnect
import IPython
from google.colab import output

display(IPython.display.Javascript('''
 function ClickConnect(){
   btn = document.querySelector("colab-connect-button")
   if (btn != null){
     console.log("Click colab-connect-button");
     btn.click()
     }

   btn = document.getElementById('ok')
   if (btn != null){
     console.log("Click reconnect");
     btn.click()
     }
  }

setInterval(ClickConnect,60000)
'''))

print("Done.")

<IPython.core.display.Javascript object>

Done.


In [110]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **Step 1: Importing Libraries**

In [111]:
import pandas as pd
import numpy as np
import seaborn as sn
import matplotlib.pyplot as plt

# For data processing
import math
from math import sqrt

# For data processing and manipulation
import matplotlib as mpl
%matplotlib inline

# For checking path
import os
import json
import joblib

#metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error

#tensorflow libs
from tensorflow import keras
from tensorflow.keras.callbacks import EarlyStopping , Callback
import tensorflow as tf
from tensorflow.keras import backend as K
from keras import backend

from tensorflow.keras.layers import *
from tensorflow.keras.layers import Dense , GRU ,Dropout , PReLU , RepeatVector ,TimeDistributed, Attention,LayerNormalization,Add,Activation
from tensorflow.keras.models import Sequential,load_model,Model
from tensorflow.keras.utils import to_categorical , plot_model
from tensorflow.keras import regularizers, constraints, initializers, activations

from sklearn.linear_model import LinearRegression
import xgboost as xgb
from lightgbm import LGBMRegressor

from tensorflow.keras.layers import InputSpec

#from keras_self_attention import SeqSelfAttention
from tensorflow.keras.layers import Concatenate

from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Bidirectional, TimeDistributed, LayerNormalization, Add
from tensorflow.keras import layers


tf.get_logger().setLevel('ERROR')
mpl.rcParams['figure.figsize'] = (8, 6)
mpl.rcParams['axes.grid'] = False

In [112]:
np.random.seed(42)

# **Step 2: Loading Necessary Dataset, Files**

**2.1 Loading Result Files**

In [113]:
weighted_avg_path = r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/DLF/Results/DLF RESULTS/weighted_average.csv'
lr_result_path = r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/DLF/Results/DLF RESULTS/linear_regression.csv'
lgbm_result_path = r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/DLF/Results/DLF RESULTS/LGBM_Regressor.csv'

In [114]:
weighted_avg = pd.read_csv(weighted_avg_path)
lr_result = pd.read_csv(lr_result_path)
lgbm_result = pd.read_csv(lgbm_result_path)


In [115]:
best_strategies = pd.read_csv(r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/DLF/Results/best_strategies.csv')
best_strategies.head(21)

,Patrol Division,bigru_test_rmse,cnn_bigru_test_rmse,simple_avg_test_rmse,weighted_avg_test_rmse,lr_test_rmse,xgb_test_rmse,lgbm_test_rmse,Best Strategy
0,1,0.713613,0.734771,0.704967,0.703575,0.688575,0.672450,0.671789,LightGBM
1,2,0.403934,0.416581,0.400836,0.399779,0.398458,0.404401,0.404372,Linear Regression
2,3,0.363826,0.392695,0.349086,0.347180,0.349630,0.352477,0.363004,Weighted Avg
3,4,0.184826,0.186770,0.180188,0.180216,0.177243,0.183516,0.183662,Linear Regression
4,5,0.194903,0.206728,0.195044,0.193526,0.193351,0.201440,0.199924,Linear Regression
5,6,0.250244,0.264965,0.250983,0.248895,0.247907,0.248468,0.247694,LightGBM
6,7,0.227689,0.298715,0.245665,0.227689,0.222979,0.236334,0.235879,Linear Regression
7,8,0.202879,0.213828,0.203509,0.201931,0.202353,0.208317,0.208260,Weighted Avg
8,9,0.197093,0.209289,0.198030,0.196184,0.196297,0.202499,0.202054,Weighted Avg
9,10,0.200406,0.207974,0.199652,0.198842,0.198224,0.204508,0.204227,Linear Regression


In [116]:
weighted_avg.head()

,Patrol Division,best_bi_gru_weight,corresponding_cnn_bi_gru_weight,best_rmse,best_mae
0,1,0.6,0.4,0.703575,0.184667
1,2,0.7,0.3,0.399779,0.091154
2,3,0.6,0.4,0.347180,0.087599
3,4,0.6,0.4,0.180216,0.054463
4,5,0.8,0.2,0.193526,0.052026


In [117]:
lr_result.head()

,Division,test rmse,test mae
0,1,0.688575,0.192168
1,2,0.398458,0.098691
2,3,0.349630,0.098854
3,4,0.177243,0.054586
4,5,0.193351,0.057960


In [118]:
lgbm_result.head()

,Division,test rmse,test mae
0,1,0.671789,0.172391
1,2,0.404372,0.081332
2,3,0.363004,0.085849
3,4,0.183662,0.039952
4,5,0.199924,0.043802


**2.2 Loading Dataset**

In [119]:
dataset_path = r'/content/drive/MyDrive/Forecasting(without weather dataset)/2.Preparing Dataset to feed Model/patrol_wise_crime_dataset'
files = os.listdir(dataset_path)

In [120]:
#loading dataset
patrol_divisons = {}
dataset = {}
for file in files:
  name = file.split('_')[0]
  id = file.split('_')[1].split('.')[0]
  patrol_divisons[int(id)] = name
  dataset[int(id)] = pd.read_csv(os.path.join(dataset_path,file))
  if 'Unnamed: 0' in dataset[int(id)].columns:
        dataset[int(id)] = dataset[int(id)].drop('Unnamed: 0', axis=1)


patrol_divisons = dict(sorted(patrol_divisons.items()))
patrol_divisons

{1: 'Central',
 2: 'Rampart',
 3: 'Southwest',
 4: 'Hollenbeck',
 5: 'Harbor',
 6: 'Hollywood',
 7: 'Wilshire',
 8: 'West LA',
 9: 'Van Nuys',
 10: 'West Valley',
 11: 'Northeast',
 12: '77th Street',
 13: 'Newton',
 14: 'Pacific',
 15: 'N Hollywood',
 16: 'Foothill',
 17: 'Devonshire',
 18: 'Southeast',
 19: 'Mission',
 20: 'Olympic',
 21: 'Topanga'}

In [121]:
dataset[1].head()

,datetime,p_id,1,2,3,4,5,6,7,8,group 0,count,day sin,day cos,week sin,week cos,year sin,year cos
0,2010-01-01 00:00:00,1,6.0,1.0,0.0,3.0,0.0,0.0,0.0,1.0,0.0,11.0,-4.416858e-12,1.000000e+00,0.781831,0.623490,0.005161,0.999987
1,2010-01-01 03:00:00,1,2.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,3.0,7.071068e-01,7.071068e-01,0.846724,0.532032,0.007311,0.999973
2,2010-01-01 06:00:00,1,0.0,0.0,0.0,12.0,1.0,0.0,0.0,1.0,0.0,14.0,1.000000e+00,6.980203e-12,0.900969,0.433884,0.009461,0.999955
3,2010-01-01 09:00:00,1,2.0,0.0,0.0,13.0,1.0,0.0,1.0,0.0,0.0,17.0,7.071068e-01,-7.071068e-01,0.943883,0.330279,0.011612,0.999933
4,2010-01-01 12:00:00,1,0.0,2.0,0.0,7.0,0.0,0.0,0.0,1.0,0.0,10.0,9.543547e-12,-1.000000e+00,0.974928,0.222521,0.013762,0.999905


**2.3 Loading Models**

In [122]:
#Loading BI GRU Models
model_path = r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/Trained_model_Files/Multi Head Attention Bi GRU/mh_attn_bi_gru_model_files'
bi_gru_models = {}
for divison in patrol_divisons.keys():
  path = os.path.join(model_path,f"mh_attn_bi_gru_all_feature_{divison}.keras")
  bi_gru_models[divison] = load_model(path)

In [123]:
#Loading CNN BI GRU Models
model_path2 =r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/Trained_model_Files/Multi Head Attention CNN Bi GRU/mh_attn_cnn_bi_gru_model_files'
cnn_bi_gru_models = {}
for divison in patrol_divisons.keys():
  path = os.path.join(model_path2,f"mh_attn_cnn_bi_gru_{divison}.keras")
  cnn_bi_gru_models[divison] = load_model(path)

**Rename Columns**

In [124]:
weighted_avg.columns

Index(['Patrol Division', 'best_bi_gru_weight',
       'corresponding_cnn_bi_gru_weight', 'best_rmse', 'best_mae'],
      dtype='object')

In [125]:
lr_result.columns

Index(['Division', 'test rmse ', 'test mae '], dtype='object')

In [126]:
lgbm_result.columns

Index(['Division', 'test rmse ', 'test mae '], dtype='object')

In [127]:
weighted_Avg = weighted_avg.copy()
weighted_Avg.rename(columns={
    'best_rmse': 'weighted_avg_test_rmse',
    'best_mae': 'weighted_avg_test_mae'
}, inplace=True)
weighted_Avg.columns

Index(['Patrol Division', 'best_bi_gru_weight',
       'corresponding_cnn_bi_gru_weight', 'weighted_avg_test_rmse',
       'weighted_avg_test_mae'],
      dtype='object')

In [128]:
lr_result.rename(columns={'Division':'Patrol Division','test rmse ':'lr_test_rmse','test mae ':'lr_test_mae'},inplace = True)
lr_result.columns

Index(['Patrol Division', 'lr_test_rmse', 'lr_test_mae'], dtype='object')

In [129]:
lgbm_result.rename(columns={'Division':'Patrol Division','test rmse ':'lgbm_test_rmse','test mae ':'lgbm_test_mae'},inplace = True)
lgbm_result.columns

Index(['Patrol Division', 'lgbm_test_rmse', 'lgbm_test_mae'], dtype='object')

# **Step 3: Predicting  for 1 Divison for each Strategy**

In [130]:
patrol_divisons = {1: 'Central',2: 'Rampart', 3: 'Southwest', 4: 'Hollenbeck', 5: 'Harbor', 6: 'Hollywood', 7: 'Wilshire', 8: 'West LA', 9: 'Van Nuys',
 10: 'West Valley', 11: 'Northeast', 12: '77th Street', 13: 'Newton', 14: 'Pacific', 15: 'N Hollywood', 16: 'Foothill', 17: 'Devonshire', 18: 'Southeast', 19: 'Mission', 20: 'Olympic', 21: 'Topanga'}

In [131]:
'''
crime_types  = {'ASSAULT': 0,
 'BURGLARY': 1,
 'CRIMINAL TRESPASS': 2,
 'DECEPTIVE PRACTICE': 3,
 'DRUG/NARCOTIC': 4,
 'HOMICIDE': 5,
 'HUMAN TRAFFICKING': 6,
 'INTERFERENCE WITH PUBLIC OFFICER': 7,
 'KIDNAPPING': 8,
 'LARCENY/THEFT': 9,
 'OTHER OFFENSES': 10,
 'PROSTITUTION': 11,
 'PUBLIC PEACE VIOLATION': 12,
 'ROBBERY': 13,
 'SEX OFFENSE': 14,
 'WEAPONS VIOLATION': 15}


crime_types  = {
    'ASSAULT' :1,
    'BURGLARY':2,
    'CRIMINAL TRESPASS':3,
    'LARENCY/THEFT'  : 4,
    'OTHER OFFENSES' : 5,
    'ROBBERY' : 6,
    'SEX OFFENSE' : 7,
    'WEAPONS VIOLATION' : 8,
    ['DECEPTIVE PRACTICE','DRUG/NARCOTIC','HOMICIDE','HUMAN TRAFFICKING','INTERFERENCE WITH PUBLIC OFFICER','KIDNAPPING','PROSTITUTION','PUBLIC PEACE VIOLATION'] : 'Group 0'
}
'''

"\ncrime_types  = {'ASSAULT': 0,\n 'BURGLARY': 1,\n 'CRIMINAL TRESPASS': 2,\n 'DECEPTIVE PRACTICE': 3,\n 'DRUG/NARCOTIC': 4,\n 'HOMICIDE': 5,\n 'HUMAN TRAFFICKING': 6,\n 'INTERFERENCE WITH PUBLIC OFFICER': 7,\n 'KIDNAPPING': 8,\n 'LARCENY/THEFT': 9,\n 'OTHER OFFENSES': 10,\n 'PROSTITUTION': 11,\n 'PUBLIC PEACE VIOLATION': 12,\n 'ROBBERY': 13,\n 'SEX OFFENSE': 14,\n 'WEAPONS VIOLATION': 15}\n\n\ncrime_types  = {\n    'ASSAULT' :1,\n    'BURGLARY':2,\n    'CRIMINAL TRESPASS':3,\n    'LARENCY/THEFT'  : 4,\n    'OTHER OFFENSES' : 5,\n    'ROBBERY' : 6,\n    'SEX OFFENSE' : 7,\n    'WEAPONS VIOLATION' : 8,\n    ['DECEPTIVE PRACTICE','DRUG/NARCOTIC','HOMICIDE','HUMAN TRAFFICKING','INTERFERENCE WITH PUBLIC OFFICER','KIDNAPPING','PROSTITUTION','PUBLIC PEACE VIOLATION'] : 'Group 0'\n}\n"

In [132]:
models_path = {
    1 : r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/DLF/DLF_Models/lgbm_models/lgb_model_1.pkl',
    2 : r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/DLF/DLF_Models/lr_models/lr_model_2.pkl',
    3 : "Weighted Average",
    4 : r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/DLF/DLF_Models/lr_models/lr_model_4.pkl',
    5 : r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/DLF/DLF_Models/lr_models/lr_model_5.pkl',
    6 : r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/DLF/DLF_Models/lgbm_models/lgb_model_6.pkl',
    7 : r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/DLF/DLF_Models/lr_models/lr_model_7.pkl',
    8 : "Weighted Average",
    9 : "Weighted Average",
    10 : r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/DLF/DLF_Models/lr_models/lr_model_10.pkl',
    11 : r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/DLF/DLF_Models/lr_models/lr_model_11.pkl',
    12 : r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/Trained_model_Files/Multi Head Attention Bi GRU/mh_attn_bi_gru_model_files/mh_attn_bi_gru_all_feature_12.keras',
    13 : r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/Trained_model_Files/Multi Head Attention Bi GRU/mh_attn_bi_gru_model_files/mh_attn_bi_gru_all_feature_13.keras',
    14 : "Weighted Average",
    15 : r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/Trained_model_Files/Multi Head Attention Bi GRU/mh_attn_bi_gru_model_files/mh_attn_bi_gru_all_feature_15.keras',
    16 : r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/DLF/DLF_Models/lgbm_models/lgb_model_16.pkl',
    17 : r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/DLF/DLF_Models/lr_models/lr_model_17.pkl',
    18 : r'/content/drive/MyDrive/Forecasting(without weather dataset)/3.Crime Forecasting/Trained_model_Files/Multi Head Attention Bi GRU/mh_attn_bi_gru_model_files/mh_attn_bi_gru_all_feature_18.keras',
    19 : "Weighted Average",
    20 : "Weighted Average",
    21 : "Weighted Average"
}

**Utility Functions**

In [133]:
def train_test_val_split(dataset):
  columns_indices = {name:i for i,name in enumerate(dataset.columns)}
  n = len(dataset)

  #splitting dataset
  training_set = dataset[:int(n*0.7)]
  validation_set = dataset[int(n*0.7):int(n*0.9)]
  test_set = dataset[int(n*0.9):]

  num_features = dataset.shape[1]
  return training_set, validation_set, test_set, num_features, columns_indices

In [134]:
class WindowGenerator():

    def __init__(self, input_width, label_width, shift,
               train_df, val_df, test_df,
               label_columns=None , shuffle=False , batch_size = 64):
        '''
        The __init__ method includes all the necessary logic for the input and label indices.
        Input:
            input_width : input width / window size
            label_width : output width
            shift : size of window shifting forward
            train_df : train dataset
            val_df : validation dataset
            test_df : test dataset
            label_columns ( Default = None) : Label Columns
            shuffle ( Default = False) : weather to shuffle data
            batch_size (Default = 64) : Batch Size
        Output: None
        Example :
            w2 = WindowGenerator(input_width=6, label_width=1, shift=1,
                     label_columns=['count'])
            w2

        '''
        # Store the raw data.
        self.train_df = train_df
        self.val_df = val_df
        self.test_df = test_df
        self.shuffle = shuffle
        self.batch_size = batch_size

        # Work out the label column indices.
        self.label_columns = label_columns
        if label_columns is not None:
            self.label_columns_indices = {name: i for i, name in
                                        enumerate(label_columns)}

        self.column_indices = {name: i for i, name in
                            enumerate(train_df.columns)}


        # Work out the window parameters.
        self.input_width = input_width
        self.label_width = label_width
        self.shift = shift

        self.total_window_size = input_width + shift

        self.input_slice = slice(0, input_width) #(start , stop)
        self.input_indices = np.arange(self.total_window_size)[self.input_slice]

        self.label_start = self.total_window_size - self.label_width
        self.labels_slice = slice(self.label_start, None)
        self.label_indices = np.arange(self.total_window_size)[self.labels_slice]

    def __repr__(self):
        return '\n'.join([
            f'Total window size: {self.total_window_size}',
            f'Input indices: {self.input_indices}',
            f'Label indices: {self.label_indices}',
            f'Label column name(s): {self.label_columns}'])

    def split_window(self, features):

        inputs = features[:, self.input_slice, :]
        labels = features[:, self.labels_slice, :]
        #taking only the labels that are presentin the label_columns
        if self.label_columns is not None:
            labels = tf.stack([labels[:, :, self.column_indices[name]] for name in self.label_columns],axis=-1)

        # Slicing doesn't preserve static shape information, so set the shapes
        # manually. This way the `tf.data.Datasets` are easier to inspect.
        inputs.set_shape([None, self.input_width, None])
        labels.set_shape([None, self.label_width, None])

        return inputs, labels




    def make_dataset(self, data):

        data = np.array(data, dtype=np.float32)
        ds = tf.keras.preprocessing.timeseries_dataset_from_array(
            data=data,
            targets=None,
            sequence_length=self.total_window_size,
            sequence_stride=1,
            shuffle=self.shuffle,
            batch_size=self.batch_size,)
        ds = ds.map(self.split_window)
        return ds

    def create_dataset2(self , map_df , reshape=True):
      x = []
      y = []
      for res in iter(map_df):
        inputs, labels = res
        if(len(inputs)==64):
          x.append(inputs)
          y.append(labels)

      x = np.array(x)
      y = np.array(y)
      if(reshape):
        x = x.reshape(-1, x.shape[-2] , x.shape[-1])
        y = y.reshape(-1 , y.shape[-2] , y.shape[-1])
      return x , y


    @property
    def train(self):
        return self.make_dataset(self.train_df)

    @property
    def val(self):
        return self.make_dataset(self.val_df)

    @property
    def test(self):
        return self.make_dataset(self.test_df)



In [135]:
def create_data(train , test , val , columns):
    '''
    Create dataset from main train , test , val with given columns
    '''
    if(columns==None):
        columns = train.columns
    new_train = train[columns]
    new_test = test[columns]
    new_val = val[columns]
    return new_train , new_test , new_val

In [136]:
#without temperature
general_indexs = ['1', '2', '3', '4', '5', '6', '7','8', 'count',
           'day sin', 'day cos', 'week sin', 'week cos',
           'year sin', 'year cos', 'group 0']

x_col = ['day sin' , 'day cos' , 'year sin' , 'year cos' , 'week cos' , 'week sin' ,'datetime']

def generate_window(df_now, ret_test = 0):
    train_df , val_df , test_df , num_features_df , column_indices_df = train_test_val_split(df_now)
    train_df , test_df , val_df = create_data(train_df , test_df , val_df , general_indexs)
    y_col = []

    #storing label columns
    for i in train_df.columns:
        if (i in x_col):
            continue
        y_col.append(i)

    #creating window generator object
    wide_window_all = WindowGenerator(train_df=train_df, test_df=test_df , val_df=val_df,
        input_width=24, label_width=24, shift=1,
        label_columns=y_col)

    if (ret_test == 1):
      return wide_window_all, test_df
    else:
      return wide_window_all


Dataset Separation(For test and validation set)

In [137]:
def separate_datset(id):
  wide_window = generate_window(dataset[id])
  test_data = wide_window.test

      # Unbatch and convert test data to numpy arrays
  x_test, y_test = [], []
  for x, y in test_data.unbatch():
          x_test.append(x.numpy())
          y_test.append(y.numpy())
  x_test = np.array(x_test)
  y_test = np.array(y_test)

  return x_test,y_test

Make Dataframe

In [138]:
def make_dataframe(y_test,predicted):
      # Number of samples you want to inspect
      num_samples = 5

      # Create a list to store the comparison DataFrames
      comparison_list = []

      # Loop over the first `num_samples`
      for i in range(num_samples):
          true_vals = y_test[i]   # shape: (24, 10)
          pred_vals = predicted[i]   # shape: (24, 10)

          # Column names for better context
          columns = ['1','2','3','4','5','6','7','8','Count','Group 0']  # Adjust if needed

          df_true = pd.DataFrame(true_vals, columns=[f"Actual_{col}" for col in columns])
          df_pred = pd.DataFrame(pred_vals, columns=[f"Predicted_{col}" for col in columns])

          df_both = pd.concat([df_true, df_pred], axis=1)
          comparison_list.append(df_both)

      # Combine all comparisons into one DataFrame
      full_comparison= pd.concat(comparison_list, axis=0)

      # Reset index
      full_comparison.reset_index(drop=True, inplace=True)

      return full_comparison

# **Forecasting with Strategy 1: Weighted Average**

In [139]:
indexs_wa = []
for key in models_path.keys():
  if models_path[key] == "Weighted Average":
    indexs_wa.append(key)

In [140]:
indexs_wa

[3, 8, 9, 14, 19, 20, 21]

In [141]:

id_1 = 3
bigru_weights = weighted_avg.loc[id_1,"best_bi_gru_weight"]
cnn_bigru_weights = weighted_avg.loc[id_1,"corresponding_cnn_bi_gru_weight"]
bi_gru_1 = bi_gru_models[id_1]
cnn_bi_gru_1 = cnn_bi_gru_models[id_1]

In [142]:
print(bigru_weights)
print(cnn_bigru_weights)

0.6
0.4


In [143]:
dataset[id_1].head()

,datetime,p_id,1,2,3,4,5,6,7,8,group 0,count,day sin,day cos,week sin,week cos,year sin,year cos
0,2010-01-01 00:00:00,3,1.0,1.0,0.0,12.0,7.0,0.0,3.0,1.0,0.0,25.0,-4.416858e-12,1.000000e+00,0.781831,0.623490,0.005161,0.999987
1,2010-01-01 03:00:00,3,2.0,1.0,1.0,0.0,4.0,0.0,1.0,0.0,0.0,9.0,7.071068e-01,7.071068e-01,0.846724,0.532032,0.007311,0.999973
2,2010-01-01 06:00:00,3,0.0,0.0,0.0,11.0,4.0,0.0,2.0,0.0,0.0,17.0,1.000000e+00,6.980203e-12,0.900969,0.433884,0.009461,0.999955
3,2010-01-01 09:00:00,3,2.0,0.0,0.0,3.0,1.0,0.0,1.0,0.0,0.0,7.0,7.071068e-01,-7.071068e-01,0.943883,0.330279,0.011612,0.999933
4,2010-01-01 12:00:00,3,0.0,2.0,0.0,25.0,9.0,1.0,6.0,0.0,0.0,43.0,9.543547e-12,-1.000000e+00,0.974928,0.222521,0.013762,0.999905


In [144]:
#taking last 24 timesteps for forecasting
x_test_1,y_test_1 = separate_datset(id_1)
last_24 = x_test_1[-1:]

In [145]:
last_24.shape

(1, 24, 16)

In [146]:
bigru_pred = bi_gru_1.predict(last_24)
cnn_bi_gru_pred = cnn_bi_gru_1.predict(last_24)

1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step


In [147]:
bigru_pred.shape

(1, 24, 10)

In [148]:
y_pred_1 = (bigru_weights * bigru_pred) + (cnn_bigru_weights * cnn_bi_gru_pred)
y_pred_1.shape

(1, 24, 10)

In [149]:
forecasted_output_1 = y_pred_1[:,-1,:]
forecasted_output_1 = np.round(forecasted_output_1)

In [150]:
forecasted_output_1

array([[ 0.,  0.,  0.,  1.,  0., -0.,  0.,  0.,  2., -0.]])

In [151]:
columns = ['1','2','3','4','5','6','7','8','Count','Group 0']
df_1 = pd.DataFrame(forecasted_output_1,columns= columns)
df_1

,1,2,3,4,5,6,7,8,Count,Group 0
0,0.0,0.0,0.0,1.0,0.0,-0.0,0.0,0.0,2.0,-0.0


# **Forecasting with Strategy 2 :Linear Regression**

In [152]:
indexs_lr = []
for key in models_path.keys():
  if models_path[key] != "Weighted Average":
    path = models_path[key]
    model_folder = path.split('/')[-2]
    if  model_folder == "lr_models":
      indexs_lr.append(key)

In [153]:
indexs_lr

[2, 4, 5, 7, 10, 11, 17]

In [154]:
id_2 = 7
ml_model = joblib.load(models_path[id_2])
bi_gru = bi_gru_models[id_2]
cnn_bi_gru = cnn_bi_gru_models[id_2]

In [155]:
dataset[id_2].head()

,datetime,p_id,1,2,3,4,5,6,7,8,group 0,count,day sin,day cos,week sin,week cos,year sin,year cos
0,2010-01-01 00:00:00,7,0.0,0.0,0.0,16.0,3.0,1.0,4.0,0.0,0.0,24.0,-4.416858e-12,1.000000e+00,0.781831,0.623490,0.005161,0.999987
1,2010-01-01 03:00:00,7,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,7.071068e-01,7.071068e-01,0.846724,0.532032,0.007311,0.999973
2,2010-01-01 06:00:00,7,1.0,0.0,0.0,12.0,3.0,0.0,1.0,1.0,0.0,18.0,1.000000e+00,6.980203e-12,0.900969,0.433884,0.009461,0.999955
3,2010-01-01 09:00:00,7,0.0,1.0,0.0,6.0,1.0,0.0,1.0,0.0,0.0,9.0,7.071068e-01,-7.071068e-01,0.943883,0.330279,0.011612,0.999933
4,2010-01-01 12:00:00,7,0.0,0.0,0.0,16.0,3.0,0.0,0.0,2.0,0.0,21.0,9.543547e-12,-1.000000e+00,0.974928,0.222521,0.013762,0.999905


In [156]:
x_test_2,y_test_2 = separate_datset(id_2)
last_24_2 = x_test_2[-1:]

In [157]:
bi_gru_pred = bi_gru.predict(last_24_2)
cnn_bi_gru_pred = cnn_bi_gru.predict(last_24_2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


In [158]:
bi_gru_pred_reshaped = bi_gru_pred.reshape(-1)
cnn_bi_gru_pred_reshaped = cnn_bi_gru_pred.reshape(-1)
X_test_stack = np.column_stack((bi_gru_pred_reshaped, cnn_bi_gru_pred_reshaped))
lr_preds = ml_model.predict(X_test_stack)


In [159]:
lr_preds = lr_preds.reshape(-1,10)
lr_preds.shape

(24, 10)

In [160]:
forecasted_output_2 = lr_preds[-1]
forecasted_output_2 = np.round(forecasted_output_2)
forecasted_output_2 = forecasted_output_2.reshape(-1,10)

In [161]:
columns  = ['1','2','3','4','5','6','7','8','Count','Group 0']
df_2 = pd.DataFrame(forecasted_output_2,columns= columns)
df_2

,1,2,3,4,5,6,7,8,Count,Group 0
0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,3.0,0.0


# **Forecasting with Strategy 3: LightGBM Regressor**

In [162]:
indexs_lgbm = []
for key in models_path.keys():
  if models_path[key] != "Weighted Average":
    path = models_path[key]
    model_folder = path.split('/')[-2]
    if  model_folder == "lgbm_models":
      indexs_lgbm.append(key)

In [163]:
indexs_lgbm

[1, 6, 16]

In [164]:
id_3 = 6
lgbm_model = joblib.load(models_path[id_3])
bi_gru = bi_gru_models[id_3]
cnn_bi_gru = cnn_bi_gru_models[id_3]

In [165]:
dataset[id_3].head()

,datetime,p_id,1,2,3,4,5,6,7,8,group 0,count,day sin,day cos,week sin,week cos,year sin,year cos
0,2010-01-01 00:00:00,6,12.0,2.0,0.0,18.0,6.0,2.0,3.0,5.0,0.0,48.0,-4.416858e-12,1.000000e+00,0.781831,0.623490,0.005161,0.999987
1,2010-01-01 03:00:00,6,3.0,0.0,0.0,2.0,1.0,0.0,0.0,0.0,0.0,6.0,7.071068e-01,7.071068e-01,0.846724,0.532032,0.007311,0.999973
2,2010-01-01 06:00:00,6,0.0,1.0,0.0,19.0,1.0,1.0,0.0,1.0,1.0,24.0,1.000000e+00,6.980203e-12,0.900969,0.433884,0.009461,0.999955
3,2010-01-01 09:00:00,6,2.0,0.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,8.0,7.071068e-01,-7.071068e-01,0.943883,0.330279,0.011612,0.999933
4,2010-01-01 12:00:00,6,1.0,0.0,0.0,18.0,1.0,0.0,0.0,2.0,0.0,22.0,9.543547e-12,-1.000000e+00,0.974928,0.222521,0.013762,0.999905


In [166]:
x_test_3,y_test_3 = separate_datset(id_3)
last_24_3 = x_test_3[-1:]

In [167]:
bi_gru_pred = bi_gru.predict(last_24_3)
cnn_bi_gru_pred = cnn_bi_gru.predict(last_24_3)

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step


In [168]:
bi_gru_pred_reshaped = bi_gru_pred.reshape(-1)
cnn_bi_gru_pred_reshaped = cnn_bi_gru_pred.reshape(-1)
X_test_stack = np.column_stack((bi_gru_pred_reshaped, cnn_bi_gru_pred_reshaped))
lgbm_preds = lgbm_model.predict(X_test_stack)

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [169]:
lgbm_preds = lgbm_preds.reshape(-1,10)
lgbm_preds.shape

(24, 10)

In [170]:
forecasted_output_3 = lgbm_preds[-1]
forecasted_output_3 = np.round(forecasted_output_3)
forecasted_output_3 = forecasted_output_3.reshape(-1,10)

In [171]:
df_3 = pd.DataFrame(forecasted_output_3,columns= columns)
df_3

,1,2,3,4,5,6,7,8,Count,Group 0
0,1.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,4.0,0.0


# **Forecasting with Strategy 4 :BI-GRU**

In [172]:
indexs_bigru = []
for key in models_path.keys():
  if models_path[key] != "Weighted Average":
    path = models_path[key]
    model_folder = path.split('/')[-2]
    if  model_folder == "mh_attn_bi_gru_model_files":
      indexs_bigru.append(key)

In [173]:
indexs_bigru

[12, 13, 15, 18]

In [174]:
id_4 = 12
bigru_model = load_model(models_path[id_4])

In [175]:
dataset[id_4].head()

,datetime,p_id,1,2,3,4,5,6,7,8,group 0,count,day sin,day cos,week sin,week cos,year sin,year cos
0,2010-01-01 00:00:00,12,2.0,2.0,0.0,16.0,8.0,1.0,3.0,2.0,0.0,34.0,-4.416858e-12,1.000000e+00,0.781831,0.623490,0.005161,0.999987
1,2010-01-01 03:00:00,12,4.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,6.0,7.071068e-01,7.071068e-01,0.846724,0.532032,0.007311,0.999973
2,2010-01-01 06:00:00,12,0.0,1.0,0.0,20.0,4.0,0.0,1.0,1.0,0.0,27.0,1.000000e+00,6.980203e-12,0.900969,0.433884,0.009461,0.999955
3,2010-01-01 09:00:00,12,0.0,1.0,0.0,7.0,2.0,0.0,0.0,2.0,0.0,12.0,7.071068e-01,-7.071068e-01,0.943883,0.330279,0.011612,0.999933
4,2010-01-01 12:00:00,12,6.0,2.0,0.0,30.0,5.0,2.0,3.0,2.0,1.0,51.0,9.543547e-12,-1.000000e+00,0.974928,0.222521,0.013762,0.999905


In [176]:
x_test_4 ,y_test_4 = separate_datset(id_4)
last_24_4 = x_test_4[-1:]

In [177]:
bigru_preds = bigru_model.predict(last_24_4)

1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step


In [178]:
bigru_preds.shape

(1, 24, 10)

In [179]:
forecasted_output_4 = bigru_preds[:,-1,:]
forecasted_output_4 = np.round(forecasted_output_4)

In [180]:
df_4 = pd.DataFrame(forecasted_output_4,columns = columns)
df_4.head()

,1,2,3,4,5,6,7,8,Count,Group 0
0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,4.0,0.0
